# Phase 4 — Parking on a Complex L-shaped Site

Tests `allocate_parking_zones` on a **6-sided L-shaped site** with three buildings,
then renders a detailed site plan showing **individual parking stalls**.

| Section | Content |
|---------|----------|
| 1 | L-shaped site + 3-building demand |
| 2 | Allocation across multiple edges |
| 3 | Detailed stall visualization (stall cells, aisle lane, dividers) |
| 4 | Shortfall scenario — demand exceeds what the L-site can hold |

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

workspace_root = Path.cwd().resolve()
candidate_roots = (
    workspace_root,
    workspace_root.parent,
    workspace_root / 'team_04',
    workspace_root.parent / 'team_04',
)
TEAM_ROOT = next((p for p in candidate_roots if (p / 'agent').exists()), None)
if TEAM_ROOT is None:
    raise FileNotFoundError('Run from workspace root, team_04/, or team_04/test_notebooks/')
if str(TEAM_ROOT) not in sys.path:
    sys.path.insert(0, str(TEAM_ROOT))
print('TEAM_ROOT:', TEAM_ROOT)

In [ ]:
import math
import plotly.graph_objects as go
from shapely.geometry import LineString, Polygon as SP

from agent.tools.parking import (
    AISLE_WIDTH_M, BAY_DEPTH_M, M2_PER_STALL,
    STALL_DEPTH_M, STALL_WIDTH_M, STRIP_INSET_M,
    allocate_parking_zones,
    compute_building_demand,
)

print('parking tool imported OK')
print(f'  stall {STALL_WIDTH_M}×{STALL_DEPTH_M} m  aisle {AISLE_WIDTH_M} m  bay depth {BAY_DEPTH_M} m')

## §1 — L-shaped site and buildings

```
(0,110)━━━━━━━━━━━━━(90,110)
   |   Building B        |
   |   [5,65→55,100]     | (inner step, north)
(0,60)            (90,60)━━━━━(150,60)
   |  Bld A               Bld C  |
   | [10,15→45,47]  [95,15→135,48]|
(0,0)━━━━━━━━━━━━━━━━━━━━━━━━━(150,0)
            Main Road (south)
```

Six boundary sides → potential parking on each edge.

In [ ]:
# L-shaped site: 150 m wide lower section + 90 m wide upper wing
SITE_BOUNDARY = [
    [0.0,   0.0,   0.0],
    [150.0, 0.0,   0.0],
    [150.0, 60.0,  0.0],
    [90.0,  60.0,  0.0],
    [90.0,  110.0, 0.0],
    [0.0,   110.0, 0.0],
    [0.0,   0.0,   0.0],
]

SITE_MODEL = {
    'boundary': SITE_BOUNDARY,
    'sides': [
        {'side_index': 0, 'start': [0.0,   0.0],   'end': [150.0, 0.0]},    # south — main road (150 m)
        {'side_index': 1, 'start': [150.0, 0.0],   'end': [150.0, 60.0]},   # east lower (60 m)
        {'side_index': 2, 'start': [150.0, 60.0],  'end': [90.0,  60.0]},   # inner notch bottom (60 m W)
        {'side_index': 3, 'start': [90.0,  60.0],  'end': [90.0,  110.0]},  # inner step east (50 m N)
        {'side_index': 4, 'start': [90.0,  110.0], 'end': [0.0,   110.0]},  # north (90 m)
        {'side_index': 5, 'start': [0.0,   110.0], 'end': [0.0,   0.0]},    # west (110 m)
    ],
    'roads': {
        'main_road_side_index': 0,
        'main_road': {'name': 'Main Road', 'width_m': 20.0},
    },
}

site_area = 150 * 60 + 90 * 50
print(f'Site: L-shaped, {site_area} m²  ({len(SITE_MODEL["sides"])} sides)')

In [ ]:
BUILDING_A = {
    'building_id': 'bld_A', 'label': 'Building A (4F)',
    'storeys': 4,
    'boundary': [
        [10.0, 15.0, 0.0], [45.0, 15.0, 0.0],
        [45.0, 47.0, 0.0], [10.0, 47.0, 0.0],
        [10.0, 15.0, 0.0],
    ],  # 35 × 32 = 1120 m²
}

BUILDING_B = {
    'building_id': 'bld_B', 'label': 'Building B (5F)',
    'storeys': 5,
    'boundary': [
        [5.0,  65.0,  0.0], [55.0, 65.0,  0.0],
        [55.0, 100.0, 0.0], [5.0,  100.0, 0.0],
        [5.0,  65.0,  0.0],
    ],  # 50 × 35 = 1750 m²
}

BUILDING_C = {
    'building_id': 'bld_C', 'label': 'Building C (3F)',
    'storeys': 3,
    'boundary': [
        [95.0,  15.0, 0.0], [135.0, 15.0, 0.0],
        [135.0, 48.0, 0.0], [95.0,  48.0, 0.0],
        [95.0,  15.0, 0.0],
    ],  # 40 × 33 = 1320 m²
}

BUILDINGS = [BUILDING_A, BUILDING_B, BUILDING_C]

demand_table = compute_building_demand(BUILDINGS)
total_stalls = sum(d['stalls_required'] for d in demand_table)

print(f'{"Building":14} {"Footprint m²":>13} {"Storeys":>7} {"Apts":>5} {"Stalls":>6}')
print('-' * 52)
fp_map = {'bld_A': 35*32, 'bld_B': 50*35, 'bld_C': 40*33}
for d in demand_table:
    b = next(b for b in BUILDINGS if b['building_id'] == d['building_id'])
    print(f'{b["label"]:14} {fp_map[d["building_id"]]:>13} {b["storeys"]:>7} {d["apartments"]:>5} {d["stalls_required"]:>6}')
print('-' * 52)
print(f'{"TOTAL":14} {"":>13} {"":>7} {sum(d["apartments"] for d in demand_table):>5} {total_stalls:>6}')

## §2 — Allocation across multiple edges

In [ ]:
SIDE_LABELS = {
    0: 'south (main road)',
    1: 'east lower',
    2: 'inner notch',
    3: 'inner step',
    4: 'north',
    5: 'west',
}

result = allocate_parking_zones(SITE_MODEL, BUILDINGS, demand_table)

print('=== ALLOCATION ===')
print(f'Stalls required : {result["stalls_required"]}')
print(f'Stalls allocated: {result["total_stalls_allocated"]}')
print(f'Shortfall       : {result["shortfall"]}')
print(f'Feasible        : {result["feasible"]}')
print(f'Summary         : {result["summary"]}')
print()
print(f'{"Zone":22} {"Edge":22} {"Main Rd":>7} {"Stalls":>6} {"Area m²":>8}')
print('-' * 70)
for z in result['zones']:
    label = SIDE_LABELS.get(z['side_index'], f'side {z["side_index"]}')
    star = '★' if z['is_main_road_side'] else ''
    print(f'{z["zone_id"]:22} {label:22} {star:>7} {z["stalls_allocated"]:>6} {z["area_sqm"]:>8.0f}')

## §3 — Detailed visualization with individual parking stalls

Each parking zone shows:
- **Yellow cells** — individual 2.5 m × 5 m stalls (alternating shade)
- **Grey strip** — 6 m aisle lane behind the stalls
- **Brown lines** — stall dividers every 2.5 m
- **Dark line** — stall / aisle separator at 5 m depth

In [ ]:
def _inward_normal(side, site_sp):
    """Return (ux, uy, nx, ny): edge unit vector and inward normal."""
    sx, sy = float(side['start'][0]), float(side['start'][1])
    ex, ey = float(side['end'][0]),   float(side['end'][1])
    dx, dy = ex - sx, ey - sy
    L = math.hypot(dx, dy)
    if L < 0.01:
        return 1.0, 0.0, 0.0, 1.0
    ux, uy = dx / L, dy / L
    nx, ny = -uy, ux
    mid_x, mid_y = (sx + ex) / 2, (sy + ey) / 2
    cx, cy = site_sp.centroid.x, site_sp.centroid.y
    if (cx - mid_x) * nx + (cy - mid_y) * ny < 0:
        nx, ny = -nx, -ny
    return ux, uy, nx, ny


def _as_simple_poly(geom):
    """Return the largest simple Polygon from a geometry, or None."""
    if geom is None or geom.is_empty:
        return None
    if isinstance(geom, SP):
        return geom
    candidates = [g for g in getattr(geom, 'geoms', [geom]) if isinstance(g, SP)]
    return max(candidates, key=lambda g: g.area) if candidates else None


def draw_stalls(fig, zone, sides, site_sp):
    """
    Draw individual stall cells, aisle shading, divider lines, and the
    stall/aisle separator line inside a single parking zone.
    """
    side_idx = zone['side_index']
    if side_idx >= len(sides):
        return

    ux, uy, nx, ny = _inward_normal(sides[side_idx], site_sp)

    # Zone polygon (2D)
    zone_poly = SP([(p[0], p[1]) for p in zone['boundary']])
    pts_2d    = [(p[0], p[1]) for p in zone['boundary']]

    # Extent in local (edge, normal) coordinates
    e_vals = [x * ux + y * uy for x, y in pts_2d]
    n_vals = [x * nx + y * ny for x, y in pts_2d]
    e_min, e_max = min(e_vals), max(e_vals)
    n_min = min(n_vals)   # near-boundary edge of the zone

    def to_xy(e, n):
        """Reconstruct absolute (x, y) from local (edge, normal) coords."""
        return e * ux + n * nx, e * uy + n * ny

    # ── 1. Aisle shading (grey rectangle from STALL_DEPTH_M to BAY_DEPTH_M) ──
    a_corners = [
        to_xy(e_min - 1, n_min + STALL_DEPTH_M),
        to_xy(e_max + 1, n_min + STALL_DEPTH_M),
        to_xy(e_max + 1, n_min + BAY_DEPTH_M + 1),
        to_xy(e_min - 1, n_min + BAY_DEPTH_M + 1),
    ]
    aisle_clip = _as_simple_poly(SP(a_corners).intersection(zone_poly))
    if aisle_clip:
        ac = list(aisle_clip.exterior.coords)
        fig.add_trace(go.Scatter(
            x=[c[0] for c in ac], y=[c[1] for c in ac],
            mode='lines', fill='toself',
            fillcolor='rgba(190,190,190,0.40)',
            line=dict(color='rgba(140,140,140,0.25)', width=0.5),
            showlegend=False, hoverinfo='skip',
        ))

    # ── 2. Stall cells (alternating fill, clipped to zone) ────────────────────
    n_cells = int((e_max - e_min) / STALL_WIDTH_M) + 1
    for i in range(n_cells):
        e_s = e_min + i * STALL_WIDTH_M
        e_e = e_s + STALL_WIDTH_M
        if e_s >= e_max + 0.05:
            break
        sc = [
            to_xy(e_s, n_min),
            to_xy(e_e, n_min),
            to_xy(e_e, n_min + STALL_DEPTH_M),
            to_xy(e_s, n_min + STALL_DEPTH_M),
        ]
        cell = _as_simple_poly(SP(sc).intersection(zone_poly))
        if cell is None or cell.area < 0.5:
            continue
        fill = 'rgba(253,224,71,0.65)' if i % 2 == 0 else 'rgba(234,179,8,0.50)'
        cc = list(cell.exterior.coords)
        fig.add_trace(go.Scatter(
            x=[c[0] for c in cc], y=[c[1] for c in cc],
            mode='lines', fill='toself',
            fillcolor=fill,
            line=dict(color='rgba(0,0,0,0)', width=0),
            showlegend=False, hoverinfo='skip',
        ))

    # ── 3. Stall divider lines (every STALL_WIDTH_M along edge) ──────────────
    for i in range(n_cells + 1):
        e_t = e_min + i * STALL_WIDTH_M
        if e_t > e_max + 0.05:
            break
        div = LineString([
            to_xy(e_t, n_min - 0.5),
            to_xy(e_t, n_min + STALL_DEPTH_M + 0.5),
        ]).intersection(zone_poly)
        if div.is_empty:
            continue
        segs = list(div.geoms) if hasattr(div, 'geoms') else [div]
        for seg in segs:
            if not hasattr(seg, 'xy') or seg.geom_type != 'LineString':
                continue
            lx, ly = seg.xy
            fig.add_trace(go.Scatter(
                x=list(lx), y=list(ly), mode='lines',
                line=dict(color='#78350f', width=1.2),
                showlegend=False, hoverinfo='skip',
            ))

    # ── 4. Stall / aisle separator line ──────────────────────────────────────
    sep = LineString([
        to_xy(e_min - 0.5, n_min + STALL_DEPTH_M),
        to_xy(e_max + 0.5, n_min + STALL_DEPTH_M),
    ]).intersection(zone_poly)
    if not sep.is_empty:
        segs = list(sep.geoms) if hasattr(sep, 'geoms') else [sep]
        for seg in segs:
            if not hasattr(seg, 'xy') or seg.geom_type != 'LineString':
                continue
            lx, ly = seg.xy
            fig.add_trace(go.Scatter(
                x=list(lx), y=list(ly), mode='lines',
                line=dict(color='#44200a', width=2.0),
                showlegend=False, hoverinfo='skip',
            ))


print('draw_stalls helper defined')

In [ ]:
def _xy_closed(pts):
    xs = [p[0] for p in pts] + [pts[0][0]]
    ys = [p[1] for p in pts] + [pts[0][1]]
    return xs, ys


BLD_COLORS = [
    ('rgba(14,116,144,0.50)', '#164e63'),   # teal
    ('rgba(109,40,217,0.45)', '#4c1d95'),   # purple
    ('rgba(5,150,105,0.45)',  '#064e3b'),   # green
]


def make_detailed_plan(site_boundary, buildings, result, sides, *, title='Parking Plan'):
    """Full site plan with buildings, zone outlines, and individual stall detail."""
    site_sp = SP([(p[0], p[1]) for p in site_boundary])
    fig = go.Figure()

    # Site boundary
    xs, ys = _xy_closed(site_boundary)
    fig.add_trace(go.Scatter(
        x=xs, y=ys, name='Site boundary',
        mode='lines', fill='toself',
        fillcolor='rgba(241,245,249,0.6)',
        line=dict(color='#1d4ed8', width=2.5),
        showlegend=True, hoverinfo='skip',
    ))

    # Buildings
    for i, bld in enumerate(buildings):
        fill, edge = BLD_COLORS[i % len(BLD_COLORS)]
        xs, ys = _xy_closed(bld['boundary'])
        fig.add_trace(go.Scatter(
            x=xs, y=ys, name=bld.get('label', f'Building {i+1}'),
            mode='lines', fill='toself',
            fillcolor=fill, line=dict(color=edge, width=1.5),
            showlegend=True, hoverinfo='skip',
        ))
        bpts = bld['boundary']
        cx = sum(p[0] for p in bpts) / len(bpts)
        cy = sum(p[1] for p in bpts) / len(bpts)
        fig.add_annotation(
            x=cx, y=cy, text=bld.get('label', ''),
            showarrow=False, font=dict(size=10, color='white', family='Arial Black'),
        )

    # Zone outlines (drawn first so stall detail goes on top)
    for z in result.get('zones', []):
        xs, ys = _xy_closed(z['boundary'])
        road_tag = ' ★' if z['is_main_road_side'] else ''
        fig.add_trace(go.Scatter(
            x=xs, y=ys,
            name=f"{z['zone_id']}{road_tag} — {z['stalls_allocated']} stalls",
            mode='lines', fill='toself',
            fillcolor='rgba(0,0,0,0)',
            line=dict(color='#b45309', width=2, dash='dash'),
            showlegend=True, hoverinfo='skip',
        ))

    # Stall detail inside each zone
    for z in result.get('zones', []):
        draw_stalls(fig, z, sides, site_sp)

    # Zone centroid labels
    for z in result.get('zones', []):
        zpts = z['boundary']
        cx = sum(p[0] for p in zpts) / len(zpts)
        cy = sum(p[1] for p in zpts) / len(zpts)
        fig.add_annotation(
            x=cx, y=cy,
            text=f"<b>P{z['stalls_allocated']}</b>",
            showarrow=False,
            font=dict(size=11, color='#7c2d12', family='Arial Black'),
            bgcolor='rgba(255,255,255,0.65)',
            borderpad=2,
        )

    # Road label
    fig.add_annotation(
        x=75, y=-6, text='▲ MAIN ROAD ▲',
        showarrow=False,
        font=dict(size=10, color='#1d4ed8'),
    )

    fig.update_layout(
        title=dict(text=title, font=dict(size=14)),
        yaxis=dict(scaleanchor='x', scaleratio=1, visible=False),
        xaxis=dict(visible=False),
        margin=dict(l=0, r=10, t=50, b=20),
        plot_bgcolor='#e8f0fe',
        paper_bgcolor='#f8fafc',
        legend=dict(x=1.01, y=1, bgcolor='white', bordercolor='#cbd5e1', borderwidth=1, font=dict(size=10)),
        width=900, height=700,
    )
    return fig


print('make_detailed_plan helper defined')

In [ ]:
fig = make_detailed_plan(
    SITE_BOUNDARY, BUILDINGS, result,
    SITE_MODEL['sides'],
    title=(
        f'L-shaped Site — {result["total_stalls_allocated"]}/{result["stalls_required"]} stalls '
        f'in {len(result["zones"])} zone(s)  |  shortfall: {result["shortfall"]}'
    ),
)
fig.show()

### Reading the stall visualization

| Element | Meaning |
|---------|----------|
| **Yellow (bright)** | Odd-numbered stalls |
| **Amber (darker)** | Even-numbered stalls |
| **Grey band** | 6 m two-way aisle |
| **Thin brown lines** | Stall dividers every 2.5 m |
| **Thick dark line** | Stall-depth boundary (5 m from zone edge) |
| **Dashed amber outline** | Zone perimeter |
| **★** | Zone fronts the main road |

## §3b — Per-zone stall detail table

In [ ]:
print(f'{"Zone":22} {"Edge":24} {"Stalls":>6} {"Area m²":>8} {"Stall rows":>10} {"Width m":>8}')
print('-' * 84)
for z in result['zones']:
    side = SITE_MODEL['sides'][z['side_index']]
    s, e = side['start'], side['end']
    side_len = math.hypot(e[0]-s[0], e[1]-s[1])
    label = SIDE_LABELS.get(z['side_index'], f'side {z["side_index"]}')
    star = ' ★' if z['is_main_road_side'] else ''
    zone_width = z['stalls_allocated'] * STALL_WIDTH_M
    print(
        f'{z["zone_id"]:22} {label+star:24} {z["stalls_allocated"]:>6} '
        f'{z["area_sqm"]:>8.0f} {"1 row":>10} {zone_width:>7.1f} m'
    )
print('-' * 84)
print(
    f'{"TOTAL":22} {"":24} {result["total_stalls_allocated"]:>6} '
    f'{sum(z["area_sqm"] for z in result["zones"]):>8.0f}'
)

## §4 — Shortfall scenario on the same L-site

Larger buildings (8F, 9F, 6F) push stall demand well above what the L-site perimeter can absorb.

In [ ]:
BIG_A = dict(BUILDING_A, building_id='big_A', label='Building A (8F)', storeys=8)
BIG_B = dict(BUILDING_B, building_id='big_B', label='Building B (9F)', storeys=9)
BIG_C = dict(BUILDING_C, building_id='big_C', label='Building C (6F)', storeys=6)
BIG_BUILDINGS = [BIG_A, BIG_B, BIG_C]

big_demand = compute_building_demand(BIG_BUILDINGS)
big_total  = sum(d['stalls_required'] for d in big_demand)
r_big      = allocate_parking_zones(SITE_MODEL, BIG_BUILDINGS, big_demand)

print(f'Stalls required : {r_big["stalls_required"]}')
print(f'Stalls allocated: {r_big["total_stalls_allocated"]}')
print(f'Shortfall       : {r_big["shortfall"]}')
print(f'Feasible        : {r_big["feasible"]}')
print(f'Summary         : {r_big["summary"]}')
print()
for z in r_big['zones']:
    label = SIDE_LABELS.get(z['side_index'], f'side {z["side_index"]}')
    print(f'  {z["zone_id"]:22} {label:22} {z["stalls_allocated"]:>4} stalls')

assert r_big['shortfall'] > 0, 'Expected a shortfall in this high-demand scenario'
print('\n✓ Shortfall correctly reported')

In [ ]:
fig_big = make_detailed_plan(
    SITE_BOUNDARY, BIG_BUILDINGS, r_big,
    SITE_MODEL['sides'],
    title=(
        f'L-site HIGH DEMAND — {r_big["total_stalls_allocated"]}/{r_big["stalls_required"]} stalls '
        f'| shortfall: {r_big["shortfall"]} stalls'
    ),
)
fig_big.show()

## Summary

| Check | Result |
|-------|--------|
| L-shaped (6-sided) site accepted | ✓ |
| Parking spans multiple edges | ✓ |
| Buildings as obstacles respected | ✓ |
| Individual stall cells drawn | ✓ |
| Aisle lane shaded | ✓ |
| Stall divider lines every 2.5 m | ✓ |
| Shortfall correctly reported | ✓ |